# Classifying Google's Open Buildings features as house or not using the trained RandomForest Model
Inputs:
- area
- type of nearest road
- distance to nearest road
- no. of buildings within 50 meters (100 meters)
- no. of POIs within 50 meters (100 meters)
- population density of city

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

In [2]:
temp = gpd.read_file("..//data//ncr//buildings.gpkg").to_crs(crs="EPSG:32651")
temp["area"] = temp.area
temp['centroid'] = temp.geometry.centroid
bldg_fin = temp.set_geometry("centroid")

In [3]:
poi_points = gpd.read_file("..//data//ncr//OSM.gpkg", layer="gis_osm_pois_free_1").to_crs(crs="EPSG:32651")
poi_poly = gpd.read_file("..//data//ncr//OSM.gpkg", layer="gis_osm_pois_a_free_1").to_crs(crs="EPSG:32651")
poi_poly['centroid'] = poi_poly.geometry.centroid
poi_cent = poi_poly.set_geometry("centroid").drop(columns=["geometry"])
poi_cent["geometry"] = poi_cent.geometry

poi_all = pd.concat([poi_points,poi_cent])

road = gpd.read_file("..//data//ncr//OSM.gpkg", layer="gis_osm_roads_free_1").to_crs(crs="EPSG:32651")
road = road.loc[~road["fclass"].isin(["footway","cycleway"])]
bldg_fin = bldg_fin.sjoin_nearest(road[["fclass","geometry"]],how="left",distance_col="dist_road",lsuffix=None,rsuffix="road").drop(columns=["index_road"])

city = gpd.read_file("..//data//ncr//adm_bounds.gpkg",layer="city").to_crs(crs="EPSG:32651")
city["density"] = (1000**2*city["pop"]/city.area).astype(int)
bldg_fin = bldg_fin.sjoin(city[["geometry","density"]],how="left",predicate="within",rsuffix=None)

In [4]:
bldg_fin_coords = np.array(list(bldg_fin.geometry.apply(lambda x: (x.x, x.y))))
poi_coords = np.array(list(poi_all.geometry.apply(lambda x: (x.x, x.y))))

btree_bldg = cKDTree(bldg_fin_coords)
btree_poi = cKDTree(poi_coords)

temp = [len(x) for x in btree_bldg.query_ball_point(bldg_fin_coords[:len(bldg_fin_coords)//2], 100)]
bldg_fin["within100m_bldg"] = temp + [len(x) for x in btree_bldg.query_ball_point(bldg_fin_coords[len(bldg_fin_coords)//2:], 100)]
temp = [len(x) for x in btree_poi.query_ball_point(bldg_fin_coords[:len(bldg_fin_coords)//2], 100)]
bldg_fin["within100m_poi"] = temp + [len(x) for x in btree_poi.query_ball_point(bldg_fin_coords[len(bldg_fin_coords)//2:], 100)]

temp = [len(x) for x in btree_bldg.query_ball_point(bldg_fin_coords[:len(bldg_fin_coords)//2], 50)]
bldg_fin["within50m_bldg"] = temp + [len(x) for x in btree_bldg.query_ball_point(bldg_fin_coords[len(bldg_fin_coords)//2:], 50)]
temp = [len(x) for x in btree_poi.query_ball_point(bldg_fin_coords[:len(bldg_fin_coords)//2], 50)]
bldg_fin["within50m_poi"] = temp + [len(x) for x in btree_poi.query_ball_point(bldg_fin_coords[len(bldg_fin_coords)//2:], 50)]

In [5]:
import pickle

with open("..//data//modeling//rfmodel.pkl","rb") as f:
    rfmodel = pickle.load(f)

def road_to_int(road_fclass):
    if road_fclass in ["trunk","trunk_link","motorway","motorway_link"]:
        return 0
    elif road_fclass in ["primary","primary_link","busway"]:
        return 1
    elif road_fclass in ["secondary","secondary_link"]:
        return 2
    elif road_fclass in ["tertiary","tertiary_link","service","unclassified"]:
        return 3
    else:
        return 4

bldg_fin["road_class"] = bldg_fin["fclass"].apply(road_to_int)

bldg_fin.drop_duplicates(subset=["full_plus_code"],inplace=True)

x_cols = ["area","road_class","dist_road","within100m_bldg","within100m_poi","within50m_bldg","within50m_poi"]
bldg_fin["bldg_class"] = rfmodel.predict(bldg_fin[x_cols])

In [6]:
bldg_fin[["full_plus_code","bldg_class"]].to_csv("..//data//ncr//bldg_class.csv",index=False)